In [ ]:
from sklearn.decomposition import PCA
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from tensorflow.keras.datasets import mnist

import tensorflow as tf
import numpy as np


Load dataset (MNIST or CIFAR-10)

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()            # Load MNIST dataset

#cifar10 = tf.keras.datasets.cifar10                                # Load CIFAR dataset - 10
#(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training data shape:", x_train.shape, y_train.shape)
print("Test data shape:", x_test.shape, y_test.shape)

y_train = np.squeeze(y_train)                                       # Convert the y array to one-dimensional - reduce the dimension of the y array 
y_test = np.squeeze(y_test)

__Preprocessing the data__

Reshape the data from 4 dimensions to 2 
Original format x_train (60000,28,28,3) : N number of images, 28 x 28 height and width
x_train.reshape(x_train.shape[0], -1) : Transform the data so that the first dimension is x_train.shape[0] (number of samples) and the second dimension contains all the remaining elements in one-dimensional form.

Normalization prepares the data for model training by converting the data to a common scale, without losing information, in order to make their analysis and comparison possible. 

float32 : conversion of data from unsigned to decimal values with a precision of 32 decimal digits, to facilitate the learning models to be created.

Divide by 255 - Normalization: 
Image data is in the range [0 255] where 0 is black and 255 is white 
By dividing each pixel value by 255, the data is converted to a scale of [0.0, 1.0]
Faster and more efficient operation of machine learning models


In [ ]:

x_train_flattened = x_train.reshape(x_train.shape[0], -1)       # Rearrange the image data in 2 dimensions
x_test_flattened = x_test.reshape(x_test.shape[0], -1)

# Normalization for training and test sets
x_train_scaled = x_train_flattened.astype('float32') / 255.0
x_test_scaled = x_test_flattened.astype('float32') / 255.0


print("Training data scaler shape:", x_train_scaled.shape, y_train.shape)
print("Test data scaler shape:", x_test_scaled.shape, y_test.shape)

Principal Components: Linear combinations of the original data features. 

With components we determine how many principal components are selected, i.e. the number of dimensions necessary to reduce computational complexity.

The goal is to find a percentage variation that will maintain a balance between the accuracy of the integrity of the data information/the accuracy of the results and the computational complexity/the training and evaluation time of the model.

PCA retains as many components as are necessary to preserve the specific percentage of the total variance in the data.  

In this project, the performance was tested and the influence of the parameter value in SVM machine learning models was evaluated for valuesn_components=0.80  
n_components=0.85  
n_components=0.90  
n_components=0.95  
n_components=0.98


In [ ]:
pca = PCA(n_components=0.85)                                                # Create PCA for dimensionality reduction

x_train_reduced = pca.fit_transform(x_train_scaled)

x_test_reduced = pca.transform(x_test_scaled)

__Linear Kernel__

Creating an SVM model using scikit-learn library 
Using linear kernel
probability = False: Disable probability prediction support for categories - less time consuming

C : normalization parameter 
    determines the model's strictness against errors 
    smaller c gives priority to a more generalized hypersurface 
    large c can lead to overfitting
    
For this project, the values applied and evaluated for the value of this specific parameter are for c = [0.01, 0.1, 0.5, 0.9, 1, 2, 5, 10, 30, 50]

The process of creating a linear model involves finding a separating hyperplane for classifying the classes.
The model that is created is initially trained on the training set in order to predict labels - the model's memory is evaluated.
The accuracy of the data is calculated and displayed - the percentage of data that was classified correctly.

np.mean() : calculates the average of the values returned by the condition (svm_linear_train_model == y_train), 
the average value equals the percentage of correct predictions since the condition can only take 2 values (True | False)

In [ ]:
svc = svm.SVC(probability = False, kernel = 'linear', C = 5)
    
svc.fit(x_train_reduced, y_train)                                           # Train the model with the training data and the unique labels

svm_linear_train_model = svc.predict(x_train_reduced)                       # Predict labels for each sample of x_train_reduced

acc_train = np.mean(svm_linear_train_model == y_train)                      # Calculating accuracy of correct predictions

print('Train Accuracy = {0:f}'.format(acc_train))    

Calculating predictions and accuracy on the test set using the trained learning model.
Using the testing data obtained after preprocessing and the preprocessing procedure (x_test_reduced).

This is new data for the algorithm that it has not seen during training.

Comparison of the set of labels predicted by the model with the set of actual labels.
The classification accuracy of the model is calculated and displayed.

In [ ]:
svm_linear_test_model = svc.predict(x_test_reduced)                          # Prediction process on new data

acc_test = np.mean(svm_linear_test_model == y_test)                          # Calculating forecast accuracy

print('Test Accuracy = {0:f}'.format(acc_test))

__Polynomial Kernel__

Creating and training an SVM model using a polynomial kernel.  
The 'poly' kernel uses polynomial functions to separate the data.  
The model is trained on the data and training labels.  

The model was tested and evaluated based on the results it returned for each of the values: C = [0.01, 0.1, 0.5, 1, 4, 5, 6, 10, 30, 50]  
When using the polynomial kernel function, the degree of the polynomial can be specified using the degree parameter.  
The polynomial degrees tested were for degree = 2, degree = 3  
probability = false    

C : check model classification errors  
degree : degree of polynomial - complexity of polynomial function

In [ ]:
svc_polynomial = svm.SVC(probability=False, kernel='poly', C=6, degree=2)        # Creating the SVC with polynomial kernel

svc_polynomial.fit(x_train_reduced, y_train)                                     # SVC Training

svm_polynomial_train_model = svc_polynomial.predict(x_train_reduced)             # Applying the model to the training set

acc_train = np.mean(svm_polynomial_train_model == y_train)                       # Calculating accuracy

print('Accuracy = {0:f}'.format(acc_train))


Subject the model to classification on unknown data (x_test_reduced).  
Generate a prediction table and store it in the svm_polynomial_test_model table.  
The svm_polynomial_test_model table contains the predicted labels in the evaluation dataset (x_test_reduced).  

Compare predictions with actual expected values - np.mean(svm_polynomial_test_model == y_test). 

Accuracy is calculated as the percentage of correct predictions made by the model (acc_test).

In [ ]:
svm_polynomial_test_model = svc_polynomial.predict(x_test_reduced)               # Predict labels for test_set

acc_test = np.mean(svm_polynomial_test_model == y_test)                          # Comparison of predicted and actual labels

print('Test Accuracy = {0:f}'.format(acc_test))


__Sigmoid Kernel__

Creating an SVM using Sigmoid kernel.  
The model is fitted to the training data (x_train_reduced, y_train).  
Predicts classes from the same dataset.  
The accuracy is calculated by comparing the predicted with the expected label values.  
The values of the parameter C tested were for C = [0.01, 0.1, 0.5, 1, 5, 6, 10, 30, 50]

In [ ]:
svm_sigmoid = svm.SVC(probability=False, kernel='sigmoid', C=30)                # Creating the SVC with sigmoid kernel

svm_sigmoid.fit(x_train_reduced, y_train)                                       # SVC Training

svm_sigmoid_train_model = svm_sigmoid.predict(x_train_reduced)                  # Calculate predictions on the training set

acc_train = np.mean(svm_sigmoid_train_model == y_train)                         # Calculating accuracy

print('Accuracy = {0:f}'.format(acc_train))


Submit the x_test_reduced dataset to evaluate the model predictions.  
Compare the model's predicted labels with the actual expected values.

In [ ]:
svm_sigmoid_test_model = svm_sigmoid.predict(x_test_reduced)                    # Calculate predictions on the test set

acc_test = np.mean(svm_sigmoid_test_model == y_test)                            # Calculating the accuracy on the test set

print('Test Accuracy = {0:f}'.format(acc_test))                                 # Print the accuracy


__RBF Kernel__

Create an SVM model using RBF Kernel.  
Train the new model on the training data (reduced_training_set)  
Calculate predictions on the training set.  
Compare labels and display percentage of correct predictions.

In [ ]:
svc_rbf = svm.SVC(probability=False, kernel='rbf', C=0.01, gamma='scale')       # Creating the SVC with rdf kernel

svc_rbf.fit(x_train_reduced, y_train)                                           # SVC Training

svm_rbf_train_model = svc_rbf.predict(x_train_reduced)                          # Calculate predictions on the training set

acc_train = np.mean(svm_rbf_train_model == y_train)                             # Calculating accuracy

print('Accuracy = {0:f}'.format(acc_train))


Calculate predictions on the evaluation dataset (x_test_reduced)   
Compare the results and calculate the percentage of correct predictions.

In [ ]:
svm_rbf_test_model = svc_rbf.predict(x_test_reduced)                    # Calculate predictions on the test set

acc_test = np.mean(svm_rbf_test_model == y_test)                        # Calculating the accuracy on the test set

print('Test Accuracy = {0:f}'.format(acc_test))                         # Print the accuracy
